In [4]:
%load_ext autoreload
%autoreload 2

In [96]:
import pandas as pd
import sys
import regression_tree as rt
import prediction_evaluation as eval
import grid_search
sys.path.append("preprocessing")
import preprocessing_food_wastage as pre_food_wastage
pd.set_option("display.max_colwidth", 500)

In [51]:
food_waste_df_train, food_waste_df_test = pre_food_wastage.preprocessing_food_wastage()

C:\Users\teoes\OneDrive\Dokument\GitHub\Machine_learning_WS_2025\A2\preprocessing\preprocessing_general.py:10: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return pd.get_dummies(df, columns=columns_for_ohe).replace({True: 1, False: 0})
C:\Users\teoes\OneDrive\Dokument\GitHub\Machine_learning_WS_2025\A2\preprocessing\preprocessing_general.py:10: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return pd.get_dummies(df, columns=columns_for_ohe).replace({True: 1, False: 0})


In [100]:
param_grid = {
    'min_samples_split': range(5,7,1),
    'max_depth': range(1,3,1),
    'split_criterion': ['sse']
}

param_grid_rf = {
    'min_samples_split': range(5,7,1),
    'max_depth': range(1,3,1),
    'split_criterion': ['sse'],
    'no_of_estimators': range(1,3,2),
    'max_features_criterion': ['sqrt']
}

In [82]:
best_tree, train_results = grid_search.grid_search_cv(food_waste_df_train, "wastage_food_amount", rt.regression_tree, rt.predict_from_tree, param_grid)
train_results

Running combo: 1
Running combo: 2
Running combo: 3
Running combo: 4


,runtime,mean_score,Parameters
0,137.442871,6.139444,"{'min_samples_split': 5, 'max_depth': 2, 'spli..."
1,141.893391,6.143657,"{'min_samples_split': 6, 'max_depth': 2, 'spli..."
2,100.414096,6.338502,"{'min_samples_split': 6, 'max_depth': 1, 'spli..."
3,114.005242,6.738300,"{'min_samples_split': 5, 'max_depth': 1, 'spli..."


In [32]:
food_waste_predictions_reg_tree = rt.predict_from_tree(best_tree, food_waste_df_test)
food_waste_rmse_reg_tree = eval.rmse(food_waste_df_test["wastage_food_amount"], food_waste_predictions_reg_tree)
print(f"RMSE score on test set: {food_waste_rmse_reg_tree}")


RMSE score on test set: 5.576393396194763


In [43]:
param_grid_scikit = {
    'min_samples_split': range(5,7,1),
    'max_depth': range(2,3,1),
    'criterion': ['squared_error']
}

sci_kit_results, best_tree_scikit = grid_search.grid_search_scikit(food_waste_df_train, "wastage_food_amount", param_grid_scikit)
sci_kit_results

Fitting 5 folds for each of 2 candidates, totalling 10 fits
DecisionTreeRegressor(max_depth=2, min_samples_split=5, random_state=1)
Time(s):  0.12425510003231466


,runtime,mean_test_score,params
0,0.124255,5.555888,"{'criterion': 'squared_error', 'max_depth': 2,..."
1,0.124255,5.555888,"{'criterion': 'squared_error', 'max_depth': 2,..."


In [44]:
food_waste_predictions_reg_tree_scikit = best_tree_scikit.predict(food_waste_df_test.loc[:, food_waste_df_test.columns != "wastage_food_amount"])
food_waste_rmse_reg_tree_scikit = eval.rmse(food_waste_df_test["wastage_food_amount"], food_waste_predictions_reg_tree_scikit)
print(f"RMSE score on test set: {food_waste_rmse_reg_tree}")
print(f"RMSE score on test set using Scikit implementation: {food_waste_rmse_reg_tree_scikit}")

RMSE score on test set: 5.576393396194763
RMSE score on test set using Scikit implementation: 5.576393396194763


In [101]:
best_forest, train_results_rf_food_waste = grid_search.grid_search_obb(food_waste_df_train, "wastage_food_amount", param_grid_rf)
train_results_rf_food_waste

Running combo: 1 out of: 4
Running combo: 2 out of: 4
Running combo: 3 out of: 4
Running combo: 4 out of: 4


,runtime,mean_score,Parameters
0,9.913246,6.775041,"{'min_samples_split': 6, 'max_depth': 2, 'split_criterion': 'sse', 'no_of_estimators': 1, 'max_features_criterion': 'sqrt'}"
1,9.791116,8.685022,"{'min_samples_split': 5, 'max_depth': 2, 'split_criterion': 'sse', 'no_of_estimators': 1, 'max_features_criterion': 'sqrt'}"
2,7.457512,10.334977,"{'min_samples_split': 5, 'max_depth': 1, 'split_criterion': 'sse', 'no_of_estimators': 1, 'max_features_criterion': 'sqrt'}"
3,8.059142,10.406545,"{'min_samples_split': 6, 'max_depth': 1, 'split_criterion': 'sse', 'no_of_estimators': 1, 'max_features_criterion': 'sqrt'}"
